# Qwen3 + Two-Tower Model for T2DM Comorbidity Prediction
## ECE176 Final Project — Colab (GPU Required)

**Architecture:**
- **Tower A (ICD Embedding):** Patient ICD code sequences → Qwen3 embedding extraction
- **Tower B (Structured Features):** 10 clinical covariates
- **Fusion:** Concatenate → MLP → Binary classification per comorbidity

**Data from local package:**
- `survival_datasets.pkl` — 26 comorbidity survival datasets (labels + covariates)
- `patient_icd_sequences.csv` — Time-ordered ICD sequences per patient
- `train_test_split.csv` — 80/20 stratified split
- `baseline_summary.json` — Baseline model results for comparison

## 0. Environment Setup

In [ ]:
!pip install -q transformers accelerate torch bitsandbytes scipy

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ★ 修改這裡指向你上傳的資料夾路徑
DATA_DIR = '/content/drive/MyDrive/colab_qwen3_package'  # <-- 改成你的路徑

import os, json, pickle
import pandas as pd
import numpy as np

# Verify files exist
for f in ['survival_datasets.pkl', 'patient_icd_sequences.csv',
          'train_test_split.csv', 'comorbidity_definitions.json']:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {'✓' if exists else '✗'} {f:45s} {size:.1f} MB")

## 1. Load Data

In [ ]:
# Load survival datasets (labels + structured features)
with open(os.path.join(DATA_DIR, 'survival_datasets.pkl'), 'rb') as f:
    survival_datasets = pickle.load(f)

# Load ICD sequences
icd_seq_df = pd.read_csv(os.path.join(DATA_DIR, 'patient_icd_sequences.csv'))

# Load train/test split
split_df = pd.read_csv(os.path.join(DATA_DIR, 'train_test_split.csv'))
train_ids = set(split_df[split_df['split'] == 'train']['subject_id'])
test_ids = set(split_df[split_df['split'] == 'test']['subject_id'])

# Load comorbidity definitions
with open(os.path.join(DATA_DIR, 'comorbidity_definitions.json'), 'r') as f:
    definitions = json.load(f)

COMORBIDITIES = list(definitions['robust_comorbidities'].keys())
COVARIATES = definitions['covariates']

# Load baseline results for comparison (if available)
baseline_path = os.path.join(DATA_DIR, 'baseline_summary.json')
if os.path.exists(baseline_path):
    with open(baseline_path, 'r') as f:
        baseline_results = json.load(f)
    print("Baseline results loaded for comparison")
else:
    baseline_results = None
    print("No baseline results found (run baseline_models_local.py first)")

print(f"\nComorbidities: {len(COMORBIDITIES)}")
print(f"Covariates: {len(COVARIATES)}")
print(f"Patients with ICD sequences: {len(icd_seq_df):,}")
print(f"Train: {len(train_ids):,} | Test: {len(test_ids):,}")
print(f"\nExample ICD sequence (first 200 chars):")
print(icd_seq_df.iloc[0]['icd_sequence'][:200])

## 2. Load Qwen3 Model

Using Qwen3-1.5B (or Qwen3-0.6B for faster inference). We extract the **last hidden state** as patient embeddings from ICD code sequences.

> If VRAM is limited (T4 = 16GB), use 4-bit quantization.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ★ 選擇模型大小（根據你的 Colab GPU VRAM 調整）
MODEL_NAME = "Qwen/Qwen3-1.7B"  # 可換成 "Qwen/Qwen3-0.6B" 如果 VRAM 不夠

# 4-bit quantization config (節省 VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EMBED_DIM = model.config.hidden_size
print(f"Model loaded! Hidden size: {EMBED_DIM}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 3. Build Prompt & Extract Embeddings

For each patient, we construct a clinical prompt from their ICD sequence and extract Qwen3's last hidden state as a dense embedding vector.

In [ ]:
def build_patient_prompt(icd_sequence_str, max_codes=80):
    """
    將 ICD 序列轉為 Qwen3 可讀的臨床摘要 prompt。

    原始格式: "V10:E119@-365|V10:I10@0|V10:N183@120|..."
    轉換為:   "Patient medical history (ICD codes in chronological order):
              Day -365: E119 (ICD-10)
              Day 0: I10 (ICD-10)
              Day 120: N183 (ICD-10)
              ..."
    """
    entries = icd_sequence_str.split('|')

    # 只取最近的 max_codes 個（避免超過 token 限制）
    if len(entries) > max_codes:
        entries = entries[-max_codes:]

    lines = []
    for entry in entries:
        try:
            ver_code, days = entry.split('@')
            ver, code = ver_code.split(':')
            ver_label = 'ICD-10' if ver == 'V10' else 'ICD-9'
            lines.append(f"  Day {days}: {code} ({ver_label})")
        except:
            continue

    prompt = (
        "Patient diagnosed with Type 2 Diabetes Mellitus (T2DM). "
        "Below is their medical history as ICD diagnosis codes in chronological order "
        "(Day 0 = T2DM diagnosis date, negative days = before T2DM):\n\n"
        + "\n".join(lines)
        + "\n\nBased on the above diagnosis history,"
    )
    return prompt


# Test with one patient
example_prompt = build_patient_prompt(icd_seq_df.iloc[0]['icd_sequence'])
print(f"Prompt length: {len(example_prompt)} chars")
print(f"Token count: {len(tokenizer.encode(example_prompt))}")
print("\n--- Example Prompt (first 500 chars) ---")
print(example_prompt[:500])

In [ ]:
@torch.no_grad()
def extract_embedding(prompt, tokenizer, model, max_length=512):
    """
    從 Qwen3 提取最後一層 hidden state 的 mean pooling 作為 embedding。
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    ).to(model.device)

    outputs = model(**inputs, output_hidden_states=True)

    # 取最後一層 hidden state，做 mean pooling
    last_hidden = outputs.hidden_states[-1]  # (1, seq_len, hidden_dim)
    attention_mask = inputs['attention_mask'].unsqueeze(-1)  # (1, seq_len, 1)
    masked = last_hidden * attention_mask
    embedding = masked.sum(dim=1) / attention_mask.sum(dim=1)  # (1, hidden_dim)

    return embedding.squeeze(0).cpu().float().numpy()


# Test
test_emb = extract_embedding(example_prompt, tokenizer, model)
print(f"Embedding shape: {test_emb.shape}")
print(f"Embedding stats: mean={test_emb.mean():.4f}, std={test_emb.std():.4f}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
import time

# ============================================================
# Batch extract embeddings for ALL patients
# ★ 這是最耗時的步驟 (~27K patients)
# ★ T4 GPU 大約 0.3-0.5 秒/patient → 3-4 小時
# ★ 建議先跑 subset 測試，確認沒問題後再跑全部
# ============================================================

# 建立 subject_id -> icd_sequence 的 mapping
icd_map = icd_seq_df.set_index('subject_id')['icd_sequence'].to_dict()

# 收集所有需要 embedding 的 patient IDs
all_patient_ids = set()
for cname, df in survival_datasets.items():
    all_patient_ids.update(df['subject_id'].tolist())
all_patient_ids = sorted(all_patient_ids & set(icd_map.keys()))

print(f"Total patients to embed: {len(all_patient_ids):,}")

# ★ 設為 True 跑全部，False 跑 subset 測試
RUN_ALL = False
if not RUN_ALL:
    SUBSET_SIZE = 500  # 先跑 500 個測試
    all_patient_ids = all_patient_ids[:SUBSET_SIZE]
    print(f"  → Running SUBSET mode: {SUBSET_SIZE} patients")

# Check if embeddings already cached
EMBED_CACHE = os.path.join(DATA_DIR, 'qwen3_embeddings.pkl')
if os.path.exists(EMBED_CACHE) and RUN_ALL:
    print("Loading cached embeddings...")
    with open(EMBED_CACHE, 'rb') as f:
        patient_embeddings = pickle.load(f)
    print(f"  Loaded {len(patient_embeddings)} embeddings")
else:
    patient_embeddings = {}
    start = time.time()

    for i, sid in enumerate(all_patient_ids):
        if sid in patient_embeddings:
            continue

        seq = icd_map.get(sid, "")
        if not seq:
            continue

        prompt = build_patient_prompt(seq)
        emb = extract_embedding(prompt, tokenizer, model)
        patient_embeddings[sid] = emb

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            remaining = (len(all_patient_ids) - i - 1) / rate
            print(f"  [{i+1}/{len(all_patient_ids)}] "
                  f"{rate:.1f} patients/sec, "
                  f"ETA: {remaining/60:.0f} min")

    elapsed = time.time() - start
    print(f"\nDone! {len(patient_embeddings)} embeddings in {elapsed/60:.1f} min")

    # Save cache
    with open(EMBED_CACHE, 'wb') as f:
        pickle.dump(patient_embeddings, f)
    print(f"Cached to {EMBED_CACHE}")

## 4. Two-Tower Fusion Model (PyTorch)

**Architecture:**
```
Tower A: Qwen3 Embedding (hidden_dim) → Linear(256) → ReLU → Dropout
Tower B: 10 Covariates             → Linear(64)  → ReLU → Dropout
                    ↓ concat (320) ↓
            Fusion: Linear(128) → ReLU → Dropout
                    Linear(1)   → Sigmoid
```

Ablation study: compare **with** vs **without** Qwen3 embeddings (Tower A only vs full model).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# ============================================================
# Dataset
# ============================================================
class ComorbidityDataset(Dataset):
    """
    每筆資料: (qwen_embedding, structured_features, label)
    """
    def __init__(self, subject_ids, surv_df, patient_embeddings,
                 covariates, horizon, embed_dim):
        self.data = []
        self.embed_dim = embed_dim

        for sid in subject_ids:
            row = surv_df[surv_df['subject_id'] == sid]
            if len(row) == 0:
                continue
            row = row.iloc[0]

            # Label: 在 horizon 年內發生共病 = 1
            label = 1.0 if (row['event'] == 1 and row['duration'] <= horizon) else 0.0

            # Structured features
            feats = [float(row[c]) for c in covariates]

            # Qwen embedding (zero vector if not available)
            emb = patient_embeddings.get(sid, np.zeros(embed_dim))

            self.data.append((emb, feats, label, sid))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        emb, feats, label, sid = self.data[idx]
        return (torch.tensor(emb, dtype=torch.float32),
                torch.tensor(feats, dtype=torch.float32),
                torch.tensor(label, dtype=torch.float32))


# ============================================================
# Two-Tower Model
# ============================================================
class TwoTowerModel(nn.Module):
    """
    Tower A: Qwen3 embedding → compress to 256-d
    Tower B: 10 structured features → expand to 64-d
    Fusion: concat(256 + 64) → 128 → 1
    """
    def __init__(self, embed_dim, n_features, use_qwen=True):
        super().__init__()
        self.use_qwen = use_qwen

        # Tower A: Qwen embedding branch
        self.tower_a = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Tower B: Structured features branch
        self.tower_b = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # Fusion
        fusion_in = 256 + 64 if use_qwen else 64
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, qwen_emb, structured):
        feat_b = self.tower_b(structured)

        if self.use_qwen:
            feat_a = self.tower_a(qwen_emb)
            fused = torch.cat([feat_a, feat_b], dim=1)
        else:
            fused = feat_b

        return self.fusion(fused).squeeze(1)


# ============================================================
# Tabular ResNet (Ablation: structured-only with skip connections)
# ============================================================
class TabularResNet(nn.Module):
    """
    帶殘差連接的 MLP，作為 ablation 對照組。
    """
    def __init__(self, n_features, hidden_dim=128, n_blocks=3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden_dim)

        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(nn.Sequential(
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden_dim, hidden_dim),
            ))

        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, qwen_emb, structured):
        # Ignores qwen_emb, only uses structured features
        x = self.input_proj(structured)
        for block in self.blocks:
            x = x + block(x)  # Skip connection
        return self.head(x).squeeze(1)


print("Models defined: TwoTowerModel, TabularResNet")

## 5. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device='cuda'):
    """Train a model and return training history + best model state."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=5, factor=0.5)

    # Class weight for imbalanced data
    all_labels = []
    for _, _, labels in train_loader:
        all_labels.append(labels)
    all_labels = torch.cat(all_labels)
    pos_weight = (len(all_labels) - all_labels.sum()) / max(all_labels.sum(), 1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

    history = {'train_loss': [], 'val_auc': [], 'val_f1': []}
    best_auc = 0
    best_state = None

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss = 0
        for emb, feats, labels in train_loader:
            emb, feats, labels = emb.to(device), feats.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(emb, feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        avg_loss = total_loss / len(train_loader.dataset)

        # --- Validate ---
        model.eval()
        all_probs, all_labels_v = [], []
        with torch.no_grad():
            for emb, feats, labels in val_loader:
                emb, feats, labels = emb.to(device), feats.to(device), labels.to(device)
                logits = model(emb, feats)
                probs = torch.sigmoid(logits)
                all_probs.append(probs.cpu())
                all_labels_v.append(labels.cpu())

        all_probs = torch.cat(all_probs).numpy()
        all_labels_v = torch.cat(all_labels_v).numpy()

        try:
            val_auc = roc_auc_score(all_labels_v, all_probs)
        except:
            val_auc = 0.5

        # Optimal F1
        prec, rec, thresholds = precision_recall_curve(all_labels_v, all_probs)
        f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
        val_f1 = f1_scores.max() if len(f1_scores) > 0 else 0

        history['train_loss'].append(avg_loss)
        history['val_auc'].append(val_auc)
        history['val_f1'].append(val_f1)

        scheduler.step(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: loss={avg_loss:.4f}, "
                  f"AUC={val_auc:.4f}, F1={val_f1:.4f}")

    # Restore best model
    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)

    return model, history, best_auc


print("Training function defined.")

## 6. Run Experiments: Ablation Study

For each comorbidity × time window, train 3 models:
1. **Two-Tower (Qwen3 + Structured)** — Full model
2. **Structured-Only MLP** — Two-Tower without Qwen3 (use_qwen=False)
3. **TabularResNet** — ResNet-style structured-only baseline

This ablation directly shows whether Qwen3 embeddings add value.

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# Main experiment loop
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ★ 選擇要跑的共病和時間窗口
# 先跑幾個高 event rate 的共病做示範
TARGET_COMORBS = ['AKI', 'IHD', 'CKD', 'Depression', 'Anxiety']  # 可改為 COMORBIDITIES 跑全部
TARGET_HORIZONS = [(3.0, '3yr'), (5.0, '5yr')]  # 可加更多

all_experiment_results = []

for cname in TARGET_COMORBS:
    surv_df = survival_datasets[cname]

    for horizon, horizon_label in TARGET_HORIZONS:
        print(f"\n{'='*60}")
        print(f"  {cname} — {horizon_label} prediction")
        print(f"{'='*60}")

        # Prepare data
        train_sids = sorted(set(surv_df['subject_id']) & train_ids & set(patient_embeddings.keys()))
        test_sids = sorted(set(surv_df['subject_id']) & test_ids & set(patient_embeddings.keys()))

        if len(train_sids) < 100 or len(test_sids) < 30:
            print(f"  SKIP: too few patients with embeddings")
            continue

        # Normalize structured features
        train_sub = surv_df[surv_df['subject_id'].isin(train_sids)]
        test_sub = surv_df[surv_df['subject_id'].isin(test_sids)]

        scaler = StandardScaler()
        scaler.fit(train_sub[COVARIATES].values)

        # Temporarily patch surv_df with scaled features
        surv_scaled = surv_df.copy()
        surv_scaled[COVARIATES] = scaler.transform(surv_df[COVARIATES].values)

        train_ds = ComorbidityDataset(train_sids, surv_scaled, patient_embeddings,
                                       COVARIATES, horizon, EMBED_DIM)
        test_ds = ComorbidityDataset(test_sids, surv_scaled, patient_embeddings,
                                      COVARIATES, horizon, EMBED_DIM)

        # Check class balance
        train_labels = [d[2].item() for d in train_ds]
        test_labels = [d[2].item() for d in test_ds]
        pos_train = sum(train_labels)
        pos_test = sum(test_labels)
        print(f"  Train: {len(train_ds)} (pos={int(pos_train)}, {pos_train/len(train_ds)*100:.1f}%)")
        print(f"  Test:  {len(test_ds)} (pos={int(pos_test)}, {pos_test/len(test_ds)*100:.1f}%)")

        if pos_train < 10 or pos_test < 5:
            print(f"  SKIP: too few positive samples")
            continue

        train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

        # --- Experiment 1: Two-Tower (Qwen3 + Structured) ---
        print("\n  [1/3] Two-Tower (Qwen3 + Structured):")
        model_tt = TwoTowerModel(EMBED_DIM, len(COVARIATES), use_qwen=True)
        model_tt, hist_tt, auc_tt = train_model(model_tt, train_loader, test_loader,
                                                  epochs=30, lr=1e-3, device=device)

        # --- Experiment 2: Structured-Only MLP ---
        print("\n  [2/3] Structured-Only MLP (no Qwen3):")
        model_no_q = TwoTowerModel(EMBED_DIM, len(COVARIATES), use_qwen=False)
        model_no_q, hist_no_q, auc_no_q = train_model(model_no_q, train_loader, test_loader,
                                                         epochs=30, lr=1e-3, device=device)

        # --- Experiment 3: TabularResNet ---
        print("\n  [3/3] TabularResNet:")
        model_resnet = TabularResNet(len(COVARIATES))
        model_resnet, hist_resnet, auc_resnet = train_model(model_resnet, train_loader, test_loader,
                                                              epochs=30, lr=1e-3, device=device)

        # Record results
        for model_name, auc_val, hist in [
            ('Two-Tower (Qwen3)', auc_tt, hist_tt),
            ('Structured-Only MLP', auc_no_q, hist_no_q),
            ('TabularResNet', auc_resnet, hist_resnet),
        ]:
            all_experiment_results.append({
                'comorbidity': cname,
                'horizon': horizon_label,
                'model': model_name,
                'best_auc': round(auc_val, 4),
                'best_f1': round(max(hist['val_f1']), 4),
            })

        # --- Plot Loss & AUC curves ---
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for name, hist, color in [
            ('Two-Tower', hist_tt, '#e74c3c'),
            ('No Qwen3', hist_no_q, '#3498db'),
            ('ResNet', hist_resnet, '#2ecc71'),
        ]:
            axes[0].plot(hist['train_loss'], label=name, color=color)
            axes[1].plot(hist['val_auc'], label=name, color=color)

        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
        axes[0].set_title(f'{cname} ({horizon_label}) — Training Loss')
        axes[0].legend()
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation AUC')
        axes[1].set_title(f'{cname} ({horizon_label}) — Validation AUC')
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        print(f"\n  SUMMARY: Two-Tower={auc_tt:.4f} | No-Qwen={auc_no_q:.4f} | ResNet={auc_resnet:.4f}")
        delta = auc_tt - auc_no_q
        print(f"  Qwen3 improvement: {delta:+.4f} ({'✓ positive' if delta > 0 else '✗ negative'})")

print("\n\nAll experiments done!")

## 7. Results Summary & Comparison with Baselines

In [ ]:
# ============================================================
# Results summary table
# ============================================================
exp_df = pd.DataFrame(all_experiment_results)
print("=== Deep Learning Model Results ===")
print(exp_df.to_string(index=False))

# Compare with baseline (if available)
if baseline_results:
    print("\n\n=== Comparison: DL Models vs Traditional Baselines ===")
    print(f"{'Comorbidity':30s} {'Horizon':8s} {'XGBoost':10s} {'LR':10s} {'Two-Tower':10s} {'Delta':10s}")
    print("-" * 78)

    for _, row in exp_df[exp_df['model'] == 'Two-Tower (Qwen3)'].iterrows():
        cname = row['comorbidity']
        horizon = row['horizon']
        tt_auc = row['best_auc']

        # Find matching baseline
        xgb_key = f'XGBoost ({horizon})'
        lr_key = f'LR ({horizon})'
        xgb_auc = baseline_results.get(cname, {}).get(xgb_key, {}).get('test', '-')
        lr_auc = baseline_results.get(cname, {}).get(lr_key, {}).get('test', '-')

        best_baseline = max(
            float(xgb_auc) if isinstance(xgb_auc, (int, float)) else 0,
            float(lr_auc) if isinstance(lr_auc, (int, float)) else 0,
        )
        delta = tt_auc - best_baseline if best_baseline > 0 else '-'

        print(f"{cname:30s} {horizon:8s} {str(xgb_auc):10s} {str(lr_auc):10s} "
              f"{tt_auc:10.4f} {str(delta) if isinstance(delta, str) else f'{delta:+.4f}':>10s}")

# Save results
exp_df.to_csv(os.path.join(DATA_DIR, 'qwen3_experiment_results.csv'),
              index=False, encoding='utf-8-sig')
print(f"\nResults saved to {DATA_DIR}/qwen3_experiment_results.csv")

## 8. Visualization: Ablation & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve

# ============================================================
# Fig 1: Ablation Study — AUC comparison bar chart
# ============================================================
if len(exp_df) > 0:
    fig, ax = plt.subplots(figsize=(14, max(6, len(exp_df) // 3 * 1.2)))

    pivot = exp_df.pivot_table(
        index=['comorbidity', 'horizon'],
        columns='model', values='best_auc')

    pivot.plot(kind='barh', ax=ax, width=0.8,
               color=['#e74c3c', '#3498db', '#2ecc71'])
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('AUC (Test Set)')
    ax.set_title('Ablation Study: Two-Tower (Qwen3) vs Structured-Only vs TabularResNet',
                 fontweight='bold')
    ax.legend(title='Model', fontsize=9)
    plt.tight_layout()
    plt.show()

# ============================================================
# Fig 2: Qwen3 improvement delta
# ============================================================
if len(exp_df) > 0:
    tt = exp_df[exp_df['model'] == 'Two-Tower (Qwen3)'].set_index(['comorbidity', 'horizon'])
    no_q = exp_df[exp_df['model'] == 'Structured-Only MLP'].set_index(['comorbidity', 'horizon'])

    deltas = (tt['best_auc'] - no_q['best_auc']).reset_index()
    deltas.columns = ['comorbidity', 'horizon', 'auc_delta']
    deltas['label'] = deltas['comorbidity'] + ' (' + deltas['horizon'] + ')'
    deltas = deltas.sort_values('auc_delta')

    fig, ax = plt.subplots(figsize=(10, max(5, len(deltas) * 0.5)))
    colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in deltas['auc_delta']]
    ax.barh(deltas['label'], deltas['auc_delta'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('AUC Improvement (Two-Tower − Structured-Only)')
    ax.set_title('Qwen3 Embedding Added Value', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Visualization complete.")